In [ ]:
import json
import os
import statistics
import numbers
import glob
from pathlib import Path
import pathlib
import shutil
import random
import yaml
import datetime

import tqdm

with open("../data_dir.json", "r") as f:
    json_data = json.load(f)
data_dir = json_data["ad_data_dir"]
config_dir = json_data["config_dir"]
if not data_dir:
    data_dir = os.path.join(os.path.dirname(os.getcwd()), "data", "ad-datasets")
else:
    data_dir = os.path.expanduser(data_dir)
if not config_dir:
    config_dir = os.path.join(os.path.dirname(os.getcwd()), "configs")
else:
    config_dir = os.path.expanduser(config_dir)
def abs_path(p):
    if os.path.isabs(p):
        return p
    return os.path.abspath(os.path.join(data_dir, p))

In [ ]:
# Summarize the contents of a JSON file containing patch metadata, i.e. statistics about numerical fields, unique values for categorical fields, and common prefixes for path fields.
def summarize_json(json_file):
    with open(abs_path(json_file), 'r') as f:
        data = json.load(f)
        header = {key: value for key, value in data.items() if key != "patches"}
        patchpath_glob = f"{os.path.commonprefix([patch['patchpath'] for patch in data.get('patches', [])])}*{os.path.splitext(data.get('patches', [{}])[0].get('patchpath', ''))[1] if len(set(data.get('patches') and os.path.splitext(data.get('patches', [{}])[0].get('patchpath', '')))) == 1 else ''}"
        def calc_stats(patches, key):
            if all(isinstance(patch, dict) for patch in patches):
                values = [patch[key] for patch in patches if key in patch and isinstance(patch[key], numbers.Number)]
            elif all(isinstance(patch, list) for patch in patches):
                values = [patch[key] for patch in patches if len(patch) > key and isinstance(patch[key], numbers.Number)]
            else:
                raise ValueError("Patches must be all dicts or all lists")
            if not values:
                return {}
            stats = {
                'count': len(values),
                'min': min(values),
                'max': max(values),
                'mean': statistics.mean(values),
                'median': statistics.median(values),
                'stdev': statistics.stdev(values) if len(values) > 1 else 0
            }
            return stats

        def unique_values(patches, key):
            if all(isinstance(patch, dict) for patch in patches):
                values = [patch[key] if not isinstance(patch[key], dict) else str(patch[key]) for patch in patches if key in patch]
            elif all(isinstance(patch, list) for patch in patches):
                values = [patch[key] if not isinstance(patch[key], dict) else str(patch[key]) for patch in patches if len(patch) > key]
            else:
                raise ValueError("Patches must be all dicts or all lists")

            unique_vals = list(set(values))
            unique_count = len(unique_vals)
            if unique_count == 1:
                unique_vals = unique_vals[0]
                return unique_vals
            if all(isinstance(patch, dict) for patch in patches):
                val_statistics = { val: sum(val==patch[key] if not isinstance(patch[key], dict) else val==str(patch[key]) for patch in patches if key in patch) for val in unique_vals[:15]}
            elif all(isinstance(patch, list) for patch in patches):
                val_statistics = { val: sum(val==patch[key] if not isinstance(patch[key], dict) else val==str(patch[key]) for patch in patches if len(patch) > key) for val in unique_vals[:15]}
            if unique_count > 15:
                val_statistics = {**val_statistics, '...': len(patches) - sum(val_statistics.values())}
            return {'unique_count': unique_count, 'unique_values': val_statistics}

        def describe_field(patches, key):
            if all(isinstance(patch, dict) for patch in patches):
                values = [patch[key] for patch in patches if key in patch]
            elif all(isinstance(patch, list) for patch in patches):
                values = [patch[key] for patch in patches if len(patch) > key]
            else:
                raise ValueError("Patches must be all dicts or all lists")

            def is_datetime_string(s):
                if not isinstance(s, str):
                    return False
                try:
                    datetime.datetime.fromisoformat(s.replace("Z", "+00:00"))
                    return True
                except ValueError:
                    return False

            if not values:
                return {}
            if all(isinstance(v, numbers.Number) for v in values):
                return calc_stats(patches, key)
            elif any(isinstance(v, (list, tuple)) for v in values):
                values = [list(v) if isinstance(v, (list, tuple)) else [v] for v in values]
                return [describe_field(values, i) for i in range(max(len(v) for v in values))]
            elif isinstance(key, str) and ("path" in key.lower() or "file" in key.lower() or "dir" in key.lower()) and all(isinstance(v, str) for v in values):
                return f"{os.path.commonprefix(values)}*"
            elif any(is_datetime_string(v) for v in values):
                dates = [datetime.datetime.fromisoformat(v.replace("Z", "+00:00")) for v in values if is_datetime_string(v)]
                stats = {
                    'count': len(dates),
                    'min': min(dates).isoformat(),
                    'max': max(dates).isoformat(),
                }
                return stats
            else:
                return unique_values(patches, key)

        patch_datas = [patch["patch_data"] for patch in data["patches"]]
        patches_summary = {
            'patchpath': patchpath_glob,
            "patch_data" : {key: describe_field(patch_datas, key) for key in set().union(*(patch.keys() for patch in patch_datas))},
        }
        summary = {**header, **patches_summary}
        return summary


In [ ]:
dataset_dirs_root = abs_path("extracted_patches")
dataset_jsons = glob.glob(os.path.join(dataset_dirs_root, "**/*.json"), recursive=True)

In [ ]:
summaries = {json_file: summarize_json(json_file) for json_file in dataset_jsons}

In [ ]:
summaries

In [ ]:
def create_subset_json(name, root_dir, parent_json, filter_func_data=None, filter_func=None, max_dataset_size=None):
    if filter_func_data is None:
        filter_func_data = lambda x: True
    if filter_func is None:
        filter_func = lambda x: True
    name_no_whitespace = name.strip().replace(" ", "_")
    subset_dir = os.path.join(root_dir, name_no_whitespace)
    os.makedirs(abs_path(subset_dir), exist_ok=True)
    with open(abs_path(parent_json), 'r') as f:
        data = json.load(f)
    filtered_patches = [patch for patch in data['patches'] if filter_func_data(patch["patch_data"]) and filter_func(patch)]
    subset_json = {
        **{key: value for key, value in data.items() if key != 'patches'},
        'patches': random.sample(filtered_patches, min(len(filtered_patches), max_dataset_size)) if max_dataset_size else filtered_patches
    }
    subset_json_path = os.path.join(subset_dir, f"{Path(parent_json).stem}_subset_{name_no_whitespace}.json")
    with open(abs_path(subset_json_path), 'w') as f:
        json.dump(subset_json, f, indent=2)
    return subset_json_path


In [ ]:
def copy_images(name, target_root_dir, subset_json):
    subset_dir = os.path.join(target_root_dir, name.strip().replace(" ", "_"), "images")
    os.makedirs(abs_path(subset_dir), exist_ok=True)
    with open(abs_path(subset_json), "r") as f:
        data = json.load(f)
    imgs = [patch["patchpath"] for patch in data["patches"]]
    for img_path in tqdm.tqdm(imgs, desc=f"Copying images to {subset_dir}"):
        if not os.path.exists(abs_path(img_path)):
            raise ValueError(f"No image at {abs_path(img_path)}.")
        shutil.copy(abs_path(img_path), abs_path(subset_dir))
    return subset_dir

In [ ]:
def create_subset(name, target_root_dir, parent_json, filter_func, max_dataset_size=None):
    subset_json = create_subset_json(name, target_root_dir, parent_json, filter_func_data=filter_func, max_dataset_size=max_dataset_size)
    img_dir = copy_images(name, target_root_dir, subset_json)
    return subset_json, img_dir

In [ ]:
def create_two_subsets(name, target_root_dir, parent_json, filter_func, max_dataset_size=None):
    subsets_dir = os.path.join(target_root_dir, f"{name}_split")
    os.makedirs(abs_path(subsets_dir), exist_ok=True)

    json1, img_dir1 = create_subset(name, subsets_dir, parent_json=parent_json, filter_func=filter_func, max_dataset_size=max_dataset_size)
    json2, img_dir2 = create_subset(f"no_{name}", subsets_dir, parent_json=parent_json, filter_func=lambda x: not filter_func(x), max_dataset_size=max_dataset_size)
    return (json1, img_dir1), (json2, img_dir2)

In [ ]:
def create_config_two_subsets(set1_name, set2_name, img_dir1, img_dir2, set1_dataset, set2_dataset = None, config_dir = config_dir, num_images_per_group = 100, purity=1.0):
    os.makedirs(config_dir, exist_ok=True)
    if set1_dataset == set2_dataset:
        set2_dataset = None
    config = {
        "data": {
            'mode': "dirs",
            'group1': set1_name,
            'group2': set2_name,
            'group1_dir': img_dir1,
            'group2_dir': img_dir2,
            'purity': purity,
            'num_images_per_group': num_images_per_group
        },
    }
    config_path = os.path.join(config_dir, f"{set1_dataset.lower()}_{set1_name.lower().replace(' ', '_')}-{f'{set2_dataset.lower()}_' if set2_dataset is not None else ''}{set2_name.lower().replace(' ', '_')}-metadata.yaml")
    with open(abs_path(config_path), 'w') as f:
        yaml.dump(config, f)
    return config_path

In [ ]:
def create_two_subsets_with_config(name, set1_name, set2_name, dataset, target_root_dir, parent_json, filter_func, max_dataset_size=None, num_images_per_group=100, purity=1.0):
    (json1, img_dir1), (json2, img_dir2) = create_two_subsets(name, target_root_dir, parent_json, filter_func, max_dataset_size)
    config_path = create_config_two_subsets(set1_name, set2_name, img_dir1, img_dir2, dataset, dataset, config_dir, num_images_per_group, purity)
    return (json1, img_dir1), (json2, img_dir2), config_path

In [ ]:
def create_subset_with_config(set1_name, set2_name, dataset, target_root_dir, parent_json, img_dir_reference_set, filter_func, max_dataset_size=None, num_images_per_group=100, purity=1.0):
    subset1_json, subset1_img_dir = create_subset(set1_name, target_root_dir, parent_json, filter_func, max_dataset_size)
    config_path = create_config_two_subsets(set1_name, set2_name, subset1_img_dir, img_dir_reference_set, dataset, dataset, config_dir=os.path.join(config_dir, target_root_dir), num_images_per_group=num_images_per_group, purity=purity)
    return (subset1_json, subset1_img_dir), (parent_json, img_dir_reference_set), config_path

In [ ]:
def change_basedir_path(json_file, new_base_dir, old_base_dir, output_json_file=None):
    with open(abs_path(json_file), 'r') as f:
        data = json.load(f)
    for patch in data.get('patches', []):
        if 'patchpath' in patch:
            patch['patchpath'] = os.path.join(new_base_dir, os.path.relpath(patch['patchpath'], old_base_dir))
    if output_json_file is None:
        output_json_file = json_file
    with open(abs_path(output_json_file), 'w') as f:
        json.dump(data, f, indent=2)
    return output_json_file

In [ ]:
create_two_subsets(
    name="nuimages_stroller",
    target_root_dir="metadata_filtered",
    parent_json="extracted_patches/nuimages/train/human/nuimages_train_human_patches.json",
    filter_func=lambda x: "stroller" in x["sub_categories"],
    max_dataset_size=100
)

In [ ]:
create_two_subsets_with_config(
    name="nuimages_stroller",
    set1_name="Pedestrians with Strollers",
    set2_name="Pedestrians without Strollers",
    dataset="nuimages",
    target_root_dir="metadata_filtered",
    parent_json="extracted_patches/nuimages/train/human/nuimages_train_human_patches.json",
    filter_func=lambda x: "stroller" in x["sub_categories"],
    max_dataset_size=100,
    num_images_per_group=50,
)

In [ ]:
create_subset_with_config(set1_name="Human with Stroller",
                          set2_name="Human",
                          dataset="nuimages",
                          target_root_dir ="metadata_filtered/splits_with_ground_truth/nuimages/human",
                          parent_json="extracted_patches/nuimages/train/human/nuimages_train_human_patches.json",
                          img_dir_reference_set="extracted_patches/nuimages/train/human/images",
                          filter_func=lambda x: "stroller" in x["sub_categories"],
                          max_dataset_size=100,
                          num_images_per_group=100,
                          purity=1.0,
                         )

In [ ]:
def create_clip_filtered_subset_jsons(clip_dir, extracted_patches_dir, pad_string):
    assert any(dataset_name in extracted_patches_dir for dataset_name in ["nuimages", "kitti", "waymo"]), "Extracted patches directory must contain one of the dataset names (nuimages, kitti, waymo) to ensure correct matching with clip image directories based on parent directory names"
    clip_image_dirs = [d for d in pathlib.Path(abs_path(clip_dir)).glob("**/") if d.is_dir() and pad_string in d.parent.name]
    extracted_patches_jsons = [p for p in pathlib.Path(abs_path(extracted_patches_dir)).glob("**/*.json") if p.is_file() and ((pad_string in p.name) if pad_string else not ("pad" in p.name))]
    clip_images_dir_to_extracted_patches_json = {
        imgs_dir: next(
            (json_file for json_file in extracted_patches_jsons if json_file.parent.name == imgs_dir.parent.parent.name),
            None
        )
        for imgs_dir in clip_image_dirs
    }
    assert all(json_file is not None for json_file in clip_images_dir_to_extracted_patches_json.values()), \
        "Not all clip image directories have a corresponding extracted patches JSON"

    jsons = []
    for imgs_dir, json_file in tqdm.tqdm(
        clip_images_dir_to_extracted_patches_json.items(),
        desc="Creating clip filtered subset JSONs"
    ):
        imgs = set(p.name for p in imgs_dir.glob("*.jpg"))
        jsons.append(create_subset_json(
            name=imgs_dir.name,
            root_dir=str(imgs_dir.parent),
            parent_json=json_file,
            filter_func=lambda x: any(Path(x["patchpath"]).name in img for img in imgs),
            max_dataset_size=None
        ))
    return jsons

In [ ]:
create_clip_filtered_subset_jsons(
    clip_dir="clip_filtered/waymo/train/vehicle",
    extracted_patches_dir="extracted_patches/waymo/train/vehicle",
    pad_string="pad0-5_clip_bbox255-0-0_th5",
)